# Customer Base Understanding

In [1]:
import pandas as pd
import numpy as np

dim_customer = pd.read_csv("../data/sql_final_set/dim_customer.csv")

dim_customer.head()


,customer_id,source_customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,...,churn,total_credit_records,avg_debt_ratio,max_90_days_late,total_late_events,ever_defaulted,customer_risk_band,late_payment_flag,credit_health_score,customer_value_segment
0,CUST000001,15634602,619,France,Female,42,2,0.00,1,1,...,1,12,162.637951,0,4,0,High Risk,1,0.0,Zero Balance
1,CUST000002,15647311,608,Spain,Female,41,1,83807.86,1,0,...,0,20,303.117201,2,9,1,High Risk,1,-10.0,Low Value
2,CUST000003,15619304,502,France,Female,42,8,159660.80,3,1,...,1,14,522.438661,1,0,1,High Risk,0,-5.0,High Value
3,CUST000004,15701354,699,France,Female,39,1,0.00,2,0,...,0,17,396.962252,0,2,1,High Risk,1,0.0,Zero Balance
4,CUST000005,15737888,850,Spain,Female,43,2,125510.82,1,1,...,0,16,464.452144,0,4,1,High Risk,1,0.0,High Value


In [2]:
# Total Customers
Total_Customers = dim_customer["customer_id"].nunique()
print("Total Customers :\n" ,Total_Customers)

Total Customers :
 10000


In [3]:
# Customer Value Distribution
Value_distribution = dim_customer["customer_value_segment"].value_counts()
print("Value Distribution :\n" ,Value_distribution)

Value Distribution :
 customer_value_segment
High Value      4799
Zero Balance    3617
Low Value       1584
Name: count, dtype: int64


In [4]:
# Customer Risk Distribution
Risk_distribution = dim_customer["customer_risk_band"].value_counts()
print("Risk Distribution:\n",Risk_distribution)

Risk Distribution:
 customer_risk_band
High Risk      7171
Medium Risk    1453
Low Risk       1376
Name: count, dtype: int64


In [5]:
# Overall Default Rate
Default_Rate = dim_customer["ever_defaulted"].mean()
print("Default Rate :\n", Default_Rate)

Default Rate :
 0.6359


In [6]:
# Default Rate by Value Segment
Default_rate_by_value_segment = dim_customer.groupby("customer_value_segment")["ever_defaulted"].mean()
print(" Default rates by Customner value segment are : \n", Default_rate_by_value_segment)

 Default rates by Customner value segment are : 
 customer_value_segment
High Value      0.632840
Low Value       0.630682
Zero Balance    0.642245
Name: ever_defaulted, dtype: float64


                                                                    ⬆️
The customer base shows a strong presence of high-value customers;
however, default rates remain consistently high across high-value, low-value, and zero-balance segments. 
This indicates that customer value alone does not explain credit risk, highlighting the need for deeper risk and behavior-based analysis.

# VALUE vs RISK MATRIX

In [7]:
# VALUE × RISK : CUSTOMER COUNT MATRIX

value_risk_count = pd.crosstab(
    dim_customer["customer_value_segment"],
    dim_customer["customer_risk_band"]
)
value_risk_count

customer_risk_band,High Risk,Low Risk,Medium Risk
customer_value_segment,,,
High Value,3462,686,651
Low Value,1106,227,251
Zero Balance,2603,463,551


In [8]:
# DEFAULT RATE MATRIX
value_risk_default = pd.crosstab(
    dim_customer["customer_value_segment"],
    dim_customer["customer_risk_band"],
    values=dim_customer["ever_defaulted"],
    aggfunc="mean"
).round(2)
value_risk_default

customer_risk_band,High Risk,Low Risk,Medium Risk
customer_value_segment,,,
High Value,0.64,0.60,0.64
Low Value,0.62,0.63,0.67
Zero Balance,0.64,0.60,0.68


In [9]:
value_risk_default_pct = value_risk_default * 100
value_risk_default_pct.round(2)


customer_risk_band,High Risk,Low Risk,Medium Risk
customer_value_segment,,,
High Value,64.0,60.0,64.0
Low Value,62.0,63.0,67.0
Zero Balance,64.0,60.0,68.0


In [10]:
# ROW-WISE SHARE
value_risk_share = value_risk_count.div(
    value_risk_count.sum(axis=1),
    axis=0
) * 100

value_risk_share.round(2)


customer_risk_band,High Risk,Low Risk,Medium Risk
customer_value_segment,,,
High Value,72.14,14.29,13.57
Low Value,69.82,14.33,15.85
Zero Balance,71.97,12.80,15.23


In [11]:
# quick sanity check 
value_risk_count.sum(axis=1), value_risk_count.sum(axis=0)


(customer_value_segment
 High Value      4799
 Low Value       1584
 Zero Balance    3617
 dtype: int64,
 customer_risk_band
 High Risk      7171
 Low Risk       1376
 Medium Risk    1453
 dtype: int64)

                                                             ⬆️

High-risk customers dominate across all value segments, 
and default rates remain consistently high within each risk band, 
indicating that customer risk classification—not customer value—is the primary driver of default behavior.

# LOAN AFFORDABILITY & CREDIT STRESS

In [12]:
import pandas as pd
import numpy as np

fact_loans = pd.read_csv("../data/sql_final_set/fact_loans.csv")
fact_loans.head()


,customer_id,SK_ID_CURR,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,AMT_INCOME_TOTAL,TARGET,loan_to_income_ratio,emi_to_income_ratio,loan_risk_flag
0,CUST007271,100002,406597.5,24700.5,351000.0,202500.0,1,2.007889,0.121978,1
1,CUST000861,100003,1293502.5,35698.5,1129500.0,270000.0,0,4.790750,0.132217,1
2,CUST005391,100004,135000.0,6750.0,135000.0,67500.0,0,2.000000,0.100000,0
3,CUST005192,100006,312682.5,29686.5,297000.0,135000.0,0,2.316167,0.219900,0
4,CUST005735,100007,513000.0,21865.5,513000.0,121500.0,0,4.222222,0.179963,1


In [13]:
# Loan to Income Ratio
fact_loans["loan_to_income_ratio"] = (
    fact_loans["AMT_CREDIT"] / fact_loans["AMT_INCOME_TOTAL"]
).round(2)

print(fact_loans["loan_to_income_ratio"].round(2).head())
 # Higher ratio = loan affordability weak.
fact_loans["loan_to_income_ratio"].describe()


0    2.01
1    4.79
2    2.00
3    2.32
4    4.22
Name: loan_to_income_ratio, dtype: float64


count    307511.000000
mean          3.957585
std           2.689684
min           0.000000
25%           2.020000
50%           3.270000
75%           5.160000
max          84.740000
Name: loan_to_income_ratio, dtype: float64

In [14]:
fact_loans["emi_to_income_ratio"] = (
    fact_loans["AMT_ANNUITY"] / fact_loans["AMT_INCOME_TOTAL"]
)
print(fact_loans["emi_to_income_ratio"].round(2).head())

# 0.4+ = EMI stress (industry thumb rule)
fact_loans["emi_to_income_ratio"].describe()


0    0.12
1    0.13
2    0.10
3    0.22
4    0.18
Name: emi_to_income_ratio, dtype: float64


count    307511.000000
mean          0.180923
std           0.094579
min           0.000000
25%           0.114776
50%           0.162833
75%           0.229064
max           1.875965
Name: emi_to_income_ratio, dtype: float64

In [15]:
# Bucket customers by EMI stress

fact_loans["emi_stress_flag"] = np.where(
    fact_loans["emi_to_income_ratio"] > 0.4, 1, 0
)
print(fact_loans["emi_stress_flag"].head())
fact_loans["emi_stress_flag"].value_counts()


0    0
1    0
2    0
3    0
4    0
Name: emi_stress_flag, dtype: int64


emi_stress_flag
0    299602
1      7909
Name: count, dtype: int64

In [16]:
# Default rate vs EMI stress
emi_stress_default = (
    fact_loans
    .groupby("emi_stress_flag")
    .agg(
        loans=("SK_ID_CURR", "count"),
        default_rate=("TARGET", "mean")
    )
    .reset_index()
)

emi_stress_default


,emi_stress_flag,loans,default_rate
0,0,299602,0.080677
1,1,7909,0.082691


                                                                    ⬆️
Most customers have manageable EMI burdens, yet default rates remain high, indicating that loan affordability is not the primary driver of defaults.
Instead, credit behavior and historical risk characteristics play a more significant role.

# PORTFOLIO RISK HEALTH & SEGMENTATION

In [17]:
# DEFAULT RATE BY RISK BAND

risk_default = (
    dim_customer
    .groupby("customer_risk_band")
    .agg(
        customers=("customer_id", "count"),
        default_rate=("ever_defaulted", "mean")
    )
    .reset_index()
)

risk_default


,customer_risk_band,customers,default_rate
0,High Risk,7171,0.637150
1,Low Risk,1376,0.603924
2,Medium Risk,1453,0.660014


In [18]:
# DEFAULT RATE BY LATE PAYMENT FLAG
late_payment_default = (
    dim_customer.groupby("late_payment_flag")
    .agg(
        customers=("customer_id","count"),
        default_rate=("ever_defaulted","mean")
    ).reset_index()
)

late_payment_default

,late_payment_flag,customers,default_rate
0,0,903,0.390919
1,1,9097,0.660218


In [19]:
# CREDIT HEALTH SCORE DISTRIBUTION
dim_customer["credit_health_score"].describe()


count    10000.000000
mean        -4.396509
std         83.921705
min       -490.000000
25%         -5.000000
50%          0.000000
75%          0.000000
max         99.875486
Name: credit_health_score, dtype: float64

In [20]:
# CREDIT HEALTH BUCKET vs DEFAULT RATE
dim_customer["credit_health_bucket"] = pd.cut(
    dim_customer["credit_health_score"],
    bins=[-100, 40, 70, 100],
    labels=["Poor", "Average", "Good"]
)

credit_health_default = (
    dim_customer
    .groupby("credit_health_bucket")
    .agg(
        customers=("customer_id", "count"),
        default_rate=("ever_defaulted", "mean")
    )
    .reset_index()
)

credit_health_default


C:\Users\unifi\AppData\Local\Temp\ipykernel_10452\3732900496.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("credit_health_bucket")


,credit_health_bucket,customers,default_rate
0,Poor,8338,0.636723
1,Average,383,0.644909
2,Good,1014,0.571006


                                                                            ⬆️
Portfolio risk analysis shows that customers with prior late payment behavior, high-risk classification,
and poor credit health scores exhibit the highest default rates, making them the most critical segments for proactive risk monitoring and control.

# Customer Transaction Behavior & Engagement

In [21]:
fact_transactions=pd.read_csv("../data/sql_final_set/fact_transactions.csv")

# BASIC TRANSACTION ACTIVITY OVERVIEW
fact_transactions.describe()


,txn_count,total_spend,avg_txn_amount,max_txn_amount,last_account_balance,spend_to_balance_ratio,high_activity_flag
count,10000.000000,1.000000e+04,10000.000000,1.000000e+04,1.000000e+04,9.973000e+03,10000.000000
mean,104.856700,1.650796e+05,1575.174272,3.979840e+04,1.256474e+05,6.275826e+03,0.249900
std,10.228716,6.851085e+04,643.305465,4.744333e+04,1.099299e+06,2.594468e+05,0.432977
min,66.000000,4.029691e+04,530.222500,3.637000e+03,0.000000e+00,4.517917e-03,0.000000
25%,98.000000,1.216801e+05,1175.941586,1.609425e+04,5.322177e+03,2.418710e+00,0.000000
50%,105.000000,1.504451e+05,1437.323542,2.654117e+04,1.825847e+04,8.330503e+00,0.000000
75%,111.000000,1.918664e+05,1812.182781,5.000000e+04,6.150905e+04,3.043516e+01,0.000000
max,144.000000,1.696514e+06,17489.836907,1.560035e+06,8.224463e+07,2.060797e+07,1.000000


In [22]:
# HIGH ACTIVITY CUSTOMER FLAG (TOP USERS)
fact_transactions["high_activity_flag"]=np.where(
    fact_transactions["txn_count"]>
    fact_transactions["txn_count"].quantile(0.75),
    1,0)

fact_transactions["high_activity_flag"].value_counts()

high_activity_flag
0    7501
1    2499
Name: count, dtype: int64

In [23]:
 # SPEND TO BALANCE RATIO (DISCIPLINE vs OVERSPENDING)
fact_transactions["spend_to_balance_ratio"] = (
    fact_transactions["total_spend"] /
    fact_transactions["last_account_balance"].replace(0, np.nan)
)

fact_transactions["spend_to_balance_ratio"].describe()


count    9.973000e+03
mean     6.275826e+03
std      2.594468e+05
min      4.517917e-03
25%      2.418710e+00
50%      8.330503e+00
75%      3.043516e+01
max      2.060797e+07
Name: spend_to_balance_ratio, dtype: float64

In [24]:
# BEHAVIOR vs DEFAULT (CUSTOMER LEVEL)
behavior_risk = (
    fact_transactions
    .merge(
        dim_customer[["customer_id", "ever_defaulted"]],
        on="customer_id",
        how="left"
    )
)
behavior_risk

,customer_id,txn_count,total_spend,avg_txn_amount,max_txn_amount,last_account_balance,spend_to_balance_ratio,high_activity_flag,ever_defaulted
0,CUST000001,110,108490.20,986.274545,19517.00,7749.14,14.000289,0,0
1,CUST000002,107,112328.32,1049.797383,15457.00,67223.16,1.670976,0,1
2,CUST000003,91,135057.39,1484.147143,26898.10,87307.83,1.546910,0,1
3,CUST000004,117,361758.15,3091.950000,100000.00,42851.45,8.442145,1,1
4,CUST000005,104,167407.62,1609.688654,19467.00,19202.24,8.718130,0,1
...,...,...,...,...,...,...,...,...,...
9995,CUST009996,134,155863.29,1163.158881,16980.00,28945.05,5.384799,1,0
9996,CUST009997,112,182593.58,1630.299821,38816.00,6578.43,27.756407,1,1
9997,CUST009998,98,137936.65,1407.516837,26346.00,362.69,380.315559,0,0
9998,CUST009999,105,102477.25,975.973810,34226.46,6901.19,14.849214,0,0


In [25]:
# DEFAULT RATE BY ACTIVITY LEVEL
activity_default = (
    behavior_risk
    .groupby("high_activity_flag")
    .agg(
        customers=("customer_id", "count"),
        default_rate=("ever_defaulted", "mean")
    )
    .reset_index()
)

activity_default


,high_activity_flag,customers,default_rate
0,0,7501,0.637782
1,1,2499,0.630252


                                                                                        ⬆️
Transaction behavior analysis shows that while most customers are actively using their accounts, overspending behavior is common.
Highly active customers exhibit slightly lower default rates,
indicating that customer engagement provides a mild but meaningful protective signal against credit risk.

# EARLY WARNING SIGNALS (Before Default Happens)

In [26]:
early_warning_df = dim_customer.copy()

fact_loans["emi_to_income_ratio"] = (
    fact_loans["AMT_ANNUITY"] / fact_loans["AMT_INCOME_TOTAL"]
)


loan_agg = (
    fact_loans
    .groupby("customer_id")
    .agg(
        avg_emi_to_income=("emi_to_income_ratio", "mean"),
        max_emi_to_income=("emi_to_income_ratio", "max")
    )
    .reset_index()
)

early_warning_df = early_warning_df.merge(
    loan_agg,
    on="customer_id",
    how="left"
)

txn_agg = (
    fact_transactions
    .groupby("customer_id")
    .agg(
        avg_spend_balance=("spend_to_balance_ratio", "mean"),
        high_activity_flag=("high_activity_flag", "max")
    )
    .reset_index()
)

early_warning_df = early_warning_df.merge(
    txn_agg,
    on="customer_id",
    how="left"
)



In [27]:
 # Late payment signal
early_warning_df["signal_late_payment"] = np.where(
    early_warning_df["late_payment_flag"] == 1, 1, 0
)

# Poor credit health signal
early_warning_df["signal_poor_credit"] = np.where(
    early_warning_df["credit_health_score"] < 40, 1, 0
)

# EMI stress signal (IMPORTANT)
early_warning_df["signal_emi_stress"] = np.where(
    early_warning_df["avg_emi_to_income"] > 0.4, 1, 0
)


# Overspending signal
early_warning_df["signal_overspend"] = np.where(
    early_warning_df["avg_spend_balance"] > 1, 1, 0
)

# Low activity signal
early_warning_df["signal_low_activity"] = np.where(
    early_warning_df["high_activity_flag"] == 0, 1, 0
)

In [28]:
 # EARLY WARNING SCORE

signal_cols = [
    "signal_late_payment",
    "signal_poor_credit",
    "signal_emi_stress",
    "signal_overspend",
    "signal_low_activity"
]

early_warning_df["early_warning_score"] = (
    early_warning_df[signal_cols].sum(axis=1)
)

early_warning_df["early_warning_score"].value_counts()



early_warning_score
4    5038
3    3874
2     977
1     108
0       3
Name: count, dtype: int64

In [29]:
# FINAL EARLY WARNING FLAG

early_warning_df["early_warning_flag"] = np.where(
    early_warning_df["early_warning_score"] >= 2,
    1, 0
)

early_warning_df["early_warning_flag"].value_counts()


early_warning_flag
1    9889
0     111
Name: count, dtype: int64

In [30]:
early_warning_df["customer_id"].nunique(), early_warning_df.shape[0]


(10000, 10000)

In [31]:
#Validation

validation = (
    early_warning_df
    .groupby("early_warning_flag")
    .agg(
        customers=("customer_id", "count"),
        default_rate=("ever_defaulted", "mean")
    )
    .reset_index()
)

validation


,early_warning_flag,customers,default_rate
0,0,111,0.459459
1,1,9889,0.637880


                                                                                   ⬆️     

The early warning framework successfully identifies high-risk customers before default by combining affordability, behavioral, 
and credit health signals, leading to significantly higher observed default rates among flagged customers.

In [34]:
dim_customer["credit_health_score"].min()

-490.0